In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv(r"C:Filepath\CleanedData.csv")


<h3>Create a Series of the taxa and their establishment means, in order of their annual totals:</h3>

In [ ]:
#Get Unique Years across all observations in descending order
years = data.loc[data["observed_on"].notna(),"observed_on"]
years = years.str[0:4].astype(int)
years = np.sort(years.unique())[::-1]

#Filter out the speciments with valid important data points: establishment means, taxon name, and observation date
graph1 = data.loc[(data["establishment_means"].notna())&(data["taxon_name"].str.contains(' '))&(data["observed_on"].notna())] 
cols = ["total_observations_08-24", "establishment_means"]
for i in years:
    cols.append(str(i))
totals = pd.DataFrame(columns=cols)


#Obtain the sum of the annual totals from 2008-2024 for each species, since data prior to that is not nearly as numerous
keywords=[]
for i in range(2008,2025):
    keywords.append(str(i))
    keyword = '|'.join(keywords)
totals["total_observations_08-24"] = graph1.loc[graph1["observed_on"].str[0:4].str.contains(keyword)]["taxon_name"].value_counts()
taxon_list = totals.index


<h3>Helper methods for data manipulation:</h3>

In [4]:
#Get the annual total for each taxon in each year.
def get_yearly_totals(df, year, taxon_list):
    yearly_counts = df.loc[(df["observed_on"].str.contains(str(year)))]["taxon_name"].value_counts()
    return yearly_counts.reindex(taxon_list, fill_value=0)
def get_establishment_means(row):
    try:
        unique_graph1 = graph1["taxon_name"].drop_duplicates(labels = "taxon_name")
        return graph1.loc[graph1["taxon_name"] == row.name, "establishment_means"].iloc[0]
    except IndexError:
        pass
#Add Subspecies data to species observations to simplify and reduce the results. Since subspecies are interbreedable, species level observations are more important.
def combine_subspecies(df):
    #part one: comine observation totals
    for row in (df.index):
        if (len(x:=df.loc[row].name.split(' '))==3) and (len(x[1])>1): 
           #if a row is a subspecies, but not a hybrid
            for i in (years):
                try:
                    df.at[f'{x[0]+" "+x[1]}',f'{i}'] = (#Sums yearly totals
                        int(df.loc[row][f'{i}']) + #subspecies
                        int(df.at[f'{x[0]+" "+x[1]}',f'{i}']))#species
                except KeyError:
                    pass 
            try:
                df.at[f'{x[0]+" "+x[1]}','total_observations'] = (#Increments total observations
                        int(df.loc[row]['total_observations']) + #subspecies
                        int(df.at[f'{x[0]+" "+x[1]}','total_observations']))#species
            except KeyError:
                pass


<h2> Combine subspecies-level data into species-level data, since subspecies are interbreedable</h2>

<h3>Sort out the establishment means of each taxa in totals</h3>

In [5]:
df_unique = graph1[['taxon_name', 'establishment_means']].drop_duplicates(subset = "taxon_name")
df_unique.set_index("taxon_name", inplace=True)
sorted_unique = df_unique.loc[taxon_list]
sorted_unique.reset_index(inplace = True)

<h3>Calculate the annual taxa observation totals, add in the establishment means, and combine each subspecies data into its species</h3>

In [6]:
for i in (years):
    totals[str(i)] = get_yearly_totals(graph1, i, taxon_list)
totals["establishment_means"] = sorted_unique["establishment_means"].values
combine_subspecies(totals)
#Subspecies data is retained, however species level data now reflects the sum of its subspecies


<h3>Now, 'Totals' contains all species and their annual observation totals, with subspecies combined.</h3>